Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\SNconsumptionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(52416, 5)

In [10]:
datosNormalizados.head(13)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 1
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 5)
Dimensiones de Y: (52404, 1)


In [15]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012]]


Se dividen nuevamente los conjuntos de datos

In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 12, 5)
Las dimensiones de testX son:  (10533, 12, 5)
Las dimensiones de valX son:  (5189, 12, 5)


In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [18]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [19]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [20]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=params['epochs'],
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [21]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

1147/1147 - 48s - 42ms/step - ia: 0.6895 - loss: 0.3408 - mae: 0.4367 - rmse: 0.5563 - smape: 0.8184 - val_ia: 0.5292 - val_loss: 0.1281 - val_mae: 0.2987 - val_rmse: 0.3393 - val_smape: 0.6548

Epoch 2/128                                           

1147/1147 - 26s - 22ms/step - ia: 0.7871 - loss: 0.1823 - mae: 0.3237 - rmse: 0.4217 - smape: 0.6236 - val_ia: 0.6354 - val_loss: 0.0622 - val_mae: 0.1983 - val_rmse: 0.2382 - val_smape: 0.4940

Epoch 3/128                                           

1147/1147 - 39s - 34ms/step - ia: 0.8149 - loss: 0.1441 - mae: 0.2846 - rmse: 0.3744 - smape: 0.5538 - val_ia: 0.6946 - val_loss: 0.0448 - val_mae: 0.1571 - val_rmse: 0.1939 - val_smape: 0.3630

Epoch 4/128                                           

1147/1147 - 35s - 30ms/step - ia: 0.8336 - loss: 0.1204 - mae: 0.2578 - rmse: 0.3421 - smape: 0.5028 - val_ia: 0.7277 - val_loss: 0.0339 - val_mae: 0.1353 - val_rmse: 0.1677 - val_smape: 0.32

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                             

144/144 - 65s - 452ms/step - ia: 0.7989 - loss: 0.1697 - mae: 0.2557 - rmse: 0.3265 - smape: 0.5798 - val_ia: 0.9361 - val_loss: 0.0131 - val_mae: 0.0836 - val_rmse: 0.1141 - val_smape: 0.2324

Epoch 2/16                                                                             

144/144 - 23s - 159ms/step - ia: 0.9649 - loss: 0.0073 - mae: 0.0580 - rmse: 0.0836 - smape: 0.1813 - val_ia: 0.9631 - val_loss: 0.0048 - val_mae: 0.0477 - val_rmse: 0.0688 - val_smape: 0.1459

Epoch 3/16                                                                             

144/144 - 24s - 165ms/step - ia: 0.9703 - loss: 0.0051 - mae: 0.0490 - rmse: 0.0698 - smape: 0.1618 - val_ia: 0.9398 - val_loss: 0.0093 - val_mae: 0.0784 - val_rmse: 0.0950 - val_smape: 0.2384

Epoch 4/16                                                                             

144/144 - 19s - 134ms/step - ia: 0.9740 - loss: 0.0041 - mae: 

...

Epoch 1/8                                                                                 

574/574 - 18s - 31ms/step - ia: 0.1809 - loss: 1.0608 - mae: 0.8523 - rmse: 1.0275 - smape: 1.5110 - val_ia: 0.2714 - val_loss: 0.9612 - val_mae: 0.8156 - val_rmse: 0.9392 - val_smape: 1.5836

Epoch 2/8                                                                                 

574/574 - 7s - 12ms/step - ia: 0.1577 - loss: 1.0350 - mae: 0.8421 - rmse: 1.0147 - smape: 1.5554 - val_ia: 0.2695 - val_loss: 0.9271 - val_mae: 0.8014 - val_rmse: 0.9240 - val_smape: 1.6284

Epoch 3/8                                                                                 

574/574 - 7s - 11ms/step - ia: 0.1400 - loss: 1.0149 - mae: 0.8342 - rmse: 1.0051 - smape: 1.6026 - val_ia: 0.2689 - val_loss: 0.8994 - val_mae: 0.7896 - val_rmse: 0.9114 - val_smape: 1.6789

Epoch 4/8                                                                                 

574/574 - 7s - 12ms/step - ia: 0.1268 - loss: 0.9992 - 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



4586/4586 - 37s - 8ms/step - ia: 0.1704 - loss: 1.3921 - mae: 0.9661 - rmse: 1.1461 - smape: 1.7506 - val_ia: 0.1515 - val_loss: 0.6635 - val_mae: 0.6749 - val_rmse: 0.6922 - val_smape: 1.4826

Epoch 2/32                                                                             

4586/4586 - 28s - 6ms/step - ia: 0.5153 - loss: 0.4772 - mae: 0.5537 - rmse: 0.6637 - smape: 1.0749 - val_ia: 0.2406 - val_loss: 0.2203 - val_mae: 0.3683 - val_rmse: 0.3854 - val_smape: 0.7811

Epoch 3/32                                                                             

4586/4586 - 42s - 9ms/step - ia: 0.7464 - loss: 0.1891 - mae: 0.3450 - rmse: 0.4203 - smape: 0.7026 - val_ia: 0.2566 - val_loss: 0.1484 - val_mae: 0.3080 - val_rmse: 0.3244 - val_smape: 0.6808

Epoch 4/32                                                                             

4586/4586 - 27s - 6ms/step - ia: 0.7982 - loss: 0.1351 - mae: 0.2874 - rmse: 0.3545 - smape: 0.6141 - val_ia: 0.2751 - val_loss: 0.1243 - val_mae: 0.28

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



287/287 - 11s - 39ms/step - ia: 0.6848 - loss: 0.2977 - mae: 0.4152 - rmse: 0.5192 - smape: 0.8309 - val_ia: 0.7746 - val_loss: 0.1665 - val_mae: 0.3227 - val_rmse: 0.3904 - val_smape: 0.6698

Epoch 2/64                                                                             

287/287 - 4s - 15ms/step - ia: 0.8544 - loss: 0.0946 - mae: 0.2346 - rmse: 0.3059 - smape: 0.5202 - val_ia: 0.8061 - val_loss: 0.1220 - val_mae: 0.2754 - val_rmse: 0.3298 - val_smape: 0.6066

Epoch 3/64                                                                             

287/287 - 4s - 13ms/step - ia: 0.8767 - loss: 0.0689 - mae: 0.1999 - rmse: 0.2614 - smape: 0.4581 - val_ia: 0.8377 - val_loss: 0.0816 - val_mae: 0.2265 - val_rmse: 0.2716 - val_smape: 0.5474

Epoch 4/64                                                                             

287/287 - 4s - 12ms/step - ia: 0.8918 - loss: 0.0535 - mae: 0.1763 - rmse: 0.2306 - smape: 0.4150 - val_ia: 0.8689 - val_loss: 0.0512 - val_mae: 0.1808 - va

In [22]:
print(best)

{'activation': 0, 'batch': 2, 'dropout': 0.1, 'epochs': 2, 'layers': 2.0, 'learning_rate': 0.0001582738817910678, 'units': 3}
